In [1]:
from dotenv import load_dotenv
from langchain_ollama import ChatOllama
import importlib.metadata as metadata
import langchain

print(f"LangChain: {langchain.__version__}")
print(f"LangGraph: {metadata.version('langgraph')}")

load_dotenv()

llm = ChatOllama(model="gpt-oss:120b-cloud", temperature=0)

print("LLM ready.")

c:\Users\ak60492\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


LangChain: 1.3.0
LangGraph: 1.2.0
LLM ready.


In [2]:
from langchain_core.tools import tool

# Reusing familiar local tools from previous lessons
@tool
def get_weather(city: str) -> str:
    """Returns the current weather for a given city."""
    return f"The weather in {city} is sunny with a high of 28C."

@tool
def get_stock_price(ticker: str) -> str:
    """Returns the current stock price for a given ticker symbol.
    Use 'ZENSAR' for Zensar Technologies, 'GOOGL' for Google."""
    prices = {"ZENSAR": "527.90 INR", "GOOGL": "175.00 USD"}
    return prices.get(ticker.upper(), f"Unknown ticker: {ticker}")

TOOLS = [get_weather, get_stock_price]
print("Tools ready: get_weather, get_stock_price")

Tools ready: get_weather, get_stock_price


In [3]:
# Example 1 - Building Smallest LangGraph
from typing import Annotated, TypedDict
from langchain_core.messages import AnyMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

# STEP 1 - Create State
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

print("AgentState is ready!!")

AgentState is ready!!


In [ ]:
# STEP 2: Define your Nodes
def call_llm(state: AgentState) -> dict:
    """One graph node: call the given LLM using current messages"""
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

In [5]:
# STEP 3: Join all nodes together using Edges
simple_graph_builder = StateGraph(AgentState)
simple_graph_builder.add_node("assistant", call_llm)

#simple_graph_builder.add_edge(first, second)
simple_graph_builder.add_edge(START, "assistant")
simple_graph_builder.add_edge("assistant", END)

simple_graph = simple_graph_builder.compile()
print('Simple graph is ready')

Simple graph is ready


In [7]:
simple_result = simple_graph.invoke({
    "messages": [{"role": "user", "content": "Explain LangGraph in one sentence"}]
})
print(simple_result["messages"][-1].content)

LangGraph is a LangChain‑based framework that lets you define, compose, and run stateful, step‑by‑step “graph” workflows of LLM agents, tools, and logic for building complex, controllable AI applications.


In [ ]:
# Example 2 - Adding tools to Conditional Edge
from langgraph.prebuilt import ToolNode

llm_with_tools = llm.bind_tools(TOOLS)

In [ ]:
# Node 1
def assistant(state: AgentState) -> dict:
    """Call the given LLM with tools"""
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

# Node 2
def route_after_assistant(state: AgentState) -> str:
    """Route to tools only when the model procuded tool calls"""
    last_message = state["messages"][-1]
    if getattr(last_message, "tool_calls", None):
        return "tools"
    return END

print("Both nodes are ready with conditional tools")